In [2]:
!pip install zss
!pip install -q pytesseract

  Preparing metadata (setup.py) ... done
  Created wheel for zss: filename=zss-1.2.0-py3-none-any.whl size=6725 sha256=147ef20bdfaf3ff629145500c2b4223ff58314bb1cdae2db3489b7727c045c14
  Stored in directory: /root/.cache/pip/wheels/46/e7/2e/44fb39352ad468427a7528cacbefefaa438a898dfd1ad2eaa4
Successfully built zss


In [1]:
!git clone https://github.com/TomasFAV/InvoiceCzech.git /content/InvoiceCzech

Cloning into '/content/InvoiceCzech'...
remote: Enumerating objects: 662, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 662 (delta 81), reused 231 (delta 73), pack-reused 398 (from 1)
Receiving objects: 100% (662/662), 40.22 MiB | 14.31 MiB/s, done.
Resolving deltas: 100% (133/133), done.
Updating files: 100% (374/374), done.


In [3]:
from InvoiceCzech.evaluation_utils.utils import *

In [4]:
a = "<pad><s_bank_account_number> 2302117393/2010</s_bank_account_number><s_bic></s_bic><s_const_symbol></s_const_symbol><s_cust_register_id> 09336273</s_cust_register_id><s_cust_tax_id> CZ09336273</s_cust_tax_id><s_due_date> 05.08.2024</s_due_date><s_iban></s_iban><s_invoice_number> 24-045023</s_invoice_number><s_issue_date> 22.07.2024</s_issue_date><s_payment_type> QR platbou</s_payment_type><s_supp_register_id> 09336273</s_supp_register_id><s_supp_tax_id> CZ09336273</s_supp_tax_id><s_taxable_supply_date></s_taxable_supply_date><s_total> 253,00</s_total><s_variable_symbol> 696711</s_variable_symbol></s>"

In [5]:
token2json(a)

{'bank_account_number': '2302117393/2010',
 'bic': [],
 'const_symbol': [],
 'cust_register_id': '09336273',
 'cust_tax_id': 'CZ09336273',
 'due_date': '05.08.2024',
 'iban': [],
 'invoice_number': '24-045023',
 'issue_date': '22.07.2024',
 'payment_type': 'QR platbou',
 'supp_register_id': '09336273',
 'supp_tax_id': 'CZ09336273',
 'taxable_supply_date': [],
 'total': '253,00',
 'variable_symbol': '696711'}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
rm -rf /content/data

In [ ]:
cp -r /content/drive/MyDrive/invoices/data/ /content/data

In [ ]:
cp -r /content/drive/MyDrive/datasets/real_validation_manually_annotated_invoices/ /content/data

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

#DATA VE FORMÁTU PRO DONUT

##KONTROLA KONZISTENCE

In [ ]:
import json

path = "/content/data/metadata_donut.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.rstrip("\n")
        if not line.strip():
            continue
        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            print("CHYBA na řádku:", i)
            print("Zpráva:", e)
            print("Pozice (0-based):", e.pos)
            start = max(0, e.pos - 120)
            end = min(len(line), e.pos + 120)
            print("\n--- okolí chyby ---")
            print(line[start:end])
            print(" " * (e.pos - start) + "^")
            break


##NAČÍTÁNÍ DAT

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage
import os, json

def load_records(data_root_folder_path: str):
    metadata_path = os.path.join(data_root_folder_path, "metadata_donut.jsonl")
    records = []

    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip() # Add this line to strip whitespace
            if not line:        # Add this check to skip empty lines
                continue
            item = json.loads(line)

            if "ground_truth" in item and "gt_parse" in item["ground_truth"]:
                gt = item["ground_truth"]["gt_parse"]
                gt.pop("vat_items", None)

                # Define key mapping
                key_mapping = {
                    "customer_register_id": "cust_register_id",
                    "customer_tax_id": "cust_tax_id",
                    "supplier_tax_id": "supp_tax_id",
                    "supplier_register_id": "supp_register_id",
                    "payment": "payment_type",
                    "total_price": "total"
                }

                # Remap keys
                for old_key, new_key in key_mapping.items():
                    if old_key in gt:
                        gt[new_key] = gt.pop(old_key)

                # Replace None values with empty strings
                for key, value in gt.items():
                    if value is None:
                        gt[key] = ""
                item["ground_truth"] = {"gt_parse": gt}

            # 3. Cesta k obrázku
            item["image"] = os.path.join(data_root_folder_path,"images",item["file_name"])

            records.append(item)

    return records

from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value, Image as HFImage

# 2. Definuj schéma datasetu (Features)
# Tohle říká datasetu: "ner_tags nejsou jen čísla, jsou to tyto konkrétní labely"
from datasets import Features, Value, Image as HFImage

features = Features({
    "file_name": Value("string"),
    "ground_truth": {
        "gt_parse": {
            "invoice_number": Value("string"),
            "supp_register_id": Value("string"),
            "supp_tax_id": Value("string"),
            "cust_register_id": Value("string"),
            "cust_tax_id": Value("string"),
            "issue_date": Value("string"),
            "taxable_supply_date": Value("string"),
            "due_date": Value("string"),
            "payment_type": Value("string"),
            "bank_account_number": Value("string"),
            "iban": Value("string"),
            "bic": Value("string"),
            "variable_symbol": Value("string"),
            "const_symbol": Value("string"),
            "total": Value("string")
        }
    },
    "image": HFImage()
})


def build_dataset(train_path: str, val_path: str | None = None):
    ds_dict = {}

    # Trénovací data
    train_records = load_records(train_path)
    # Při vytváření Datasetu rovnou vnutíme features
    train_ds = Dataset.from_list(train_records, features=features)
    ds_dict["train"] = train_ds

    # Validační data
    if val_path is not None:
        val_records = load_records(val_path)
        val_ds = Dataset.from_list(val_records, features=features)
        ds_dict["val"] = val_ds

    return DatasetDict(ds_dict)

# ---------- použití ----------

train_root = "/content/data/"

dataset = build_dataset(train_root)

print(dataset)
print(dataset["train"][0])

if "val" in dataset:
    print(dataset["val"][0])

DatasetDict({
    train: Dataset({
        features: ['file_name', 'ground_truth', 'image'],
        num_rows: 39
    })
})
{'file_name': 'manually_labeled_2026-02-15_17-12-42.png', 'ground_truth': {'gt_parse': {'invoice_number': '', 'supp_register_id': '26043319', 'supp_tax_id': 'CZ26043319', 'cust_register_id': '', 'cust_tax_id': '', 'issue_date': '24.12.2025', 'taxable_supply_date': '', 'due_date': '3.1.2026', 'payment_type': '', 'bank_account_number': '830809001/5500', 'iban': 'CZ5955000000000830809001', 'bic': 'RZBCCZPP', 'variable_symbol': '5251227361', 'const_symbol': '308', 'total': '441,65'}}, 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1654x2339 at 0x7DA89414A450>}


#DATA VE FORMÁTU LAYOUTLMV3

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage
import os, json

KEEP_KEYS = {
    "invoice_number",
    "issue_date",
    "due_date",
    "taxable_supply_date",
}

def filter_gt(obj, keep_keys: set):
    if not isinstance(obj, dict):
        return {}
    filtered = {}
    for k in keep_keys:
        if k in obj:
            v = obj[k]
            filtered[k] = "" if v is None else v
    return filtered

def load_records(data_root_folder_path: str):
    metadata_path = os.path.join(data_root_folder_path, "metadata_layoutlmv3.jsonl")
    records = []

    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)

            # 1. Ponecháme v "data" jen boxy a slova (tokens)
            # Přistupujeme k cestě: item -> data -> tokens
            if "data" in item and "tokens" in item["data"]:
                relevant_data = {
                    "bboxes": item["data"]["tokens"].get("boxes", []),
                    "tokens": item["data"]["tokens"].get("tokens", []),
                }

                tags = []

                for i in range(len(item["data"]["tokens"].get("boxes", []))):
                    tag = item["data"]["tokens"].get("tags", [])[i]

                    item["data"]["tokens"]["boxes"][i][0] = int(item["data"]["tokens"]["boxes"][i][0])
                    item["data"]["tokens"]["boxes"][i][1] = int(item["data"]["tokens"]["boxes"][i][1])
                    item["data"]["tokens"]["boxes"][i][2] = int(item["data"]["tokens"]["boxes"][i][2])
                    item["data"]["tokens"]["boxes"][i][3] = int(item["data"]["tokens"]["boxes"][i][3])

                    # 1. Odstranění PAYMENT_TYPE (17, 18) a všech VAT polí (29, 30, 31, 32, 33, 34)
                    if tag in [31, 32, 33, 34]:
                        tag = 0

                    tags.append(tag)

                item.pop("data", None)

                item["ner_tags"] = tags
                item["bboxes"] = relevant_data["bboxes"]
                item["tokens"] = relevant_data["tokens"]

            # 2. Ground Truth - ponecháme jen gt_parse (očištěný o vat_items)
            if "ground_truth" in item and "gt_parse" in item["ground_truth"]:
                gt = item["ground_truth"]["gt_parse"]
                gt.pop("vat_items", None)

                # Define key mapping
                key_mapping = {
                    "customer_register_id": "cust_register_id",
                    "customer_tax_id": "cust_tax_id",
                    "supplier_tax_id": "supp_tax_id",
                    "supplier_register_id": "supp_register_id",
                    "payment": "payment_type",
                    "total_price": "total"
                }

                # Remap keys
                for old_key, new_key in key_mapping.items():
                    if old_key in gt:
                        gt[new_key] = gt.pop(old_key)

                # Replace None values with empty strings
                for key, value in gt.items():
                    if value is None:
                        gt[key] = ""
                item["ground_truth"] = {"gt_parse": gt}

            # 3. Cesta k obrázku
            item["image"] = os.path.join(data_root_folder_path,"images", item["file_name"])

            records.append(item)

    return records

from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value, Image as HFImage

# 1. Definuj seznam labelů přesně v tom pořadí, jak jdou ID (0 až 36)
# Pokud jsi některá ID vynechal (přeindexoval), musíš seznam upravit,
# aby délka a indexy seděly.
label_names = [
    "O",                             # 0
    "B_INVOICE_NUMBER", "I_INVOICE_NUMBER",   # 1, 2
    "B_SUPP_REGISTER_ID", "I_SUPP_REGISTER_ID", # 3, 4
    "B_SUPP_TAX_ID", "I_SUPP_TAX_ID",           # 5, 6
    "B_CUST_REGISTER_ID", "I_CUST_REGISTER_ID", # 7, 8
    "B_CUST_TAX_ID", "I_CUST_TAX_ID",           # 9, 10
    "B_ISSUE_DATE", "I_ISSUE_DATE",                     # 11, 12
    "B_TAXABLE_SUPPLY_DATE", "I_TAXABLE_SUPPLY_DATE",   # 13, 14
    "B_DUE_DATE", "I_DUE_DATE",                         # 15, 16
    "B_PAYMENT_TYPE", "I_PAYMENT_TYPE",                 # 17, 18
    "B_BANK_ACCOUNT_NUMBER", "I_BANK_ACCOUNT_NUMBER",   # 19, 20
    "B_IBAN", "I_IBAN",                                 # 21, 22
    "B_BIC", "I_BIC",                                   # 23, 24
    "B_VARIABLE_SYMBOL", "I_VARIABLE_SYMBOL",           # 25, 26
    "B_CONST_SYMBOL", "I_CONST_SYMBOL",                 # 27, 28
    "B_TOTAL", "I_TOTAL"
    # Zbytek do 36 můžeš nechat jako placeholder, pokud je nepoužíváš,
    # nebo seznam zkrať, pokud jsi ID posunul úplně.
]

# 2. Definuj schéma datasetu (Features)
# Tohle říká datasetu: "ner_tags nejsou jen čísla, jsou to tyto konkrétní labely"
features = Features({
    "file_name": Value("string"),
    "tokens": Sequence(Value("string")),
    "bboxes": Sequence(Sequence(Value("int64"))),
    "ner_tags": Sequence(ClassLabel(names=label_names)),
    "image": HFImage(),
})

def build_dataset(train_path: str, val_path: str | None = None):
    ds_dict = {}

    # Trénovací data
    train_records = load_records(train_path)
    # Při vytváření Datasetu rovnou vnutíme features
    train_ds = Dataset.from_list(train_records, features=features)
    ds_dict["train"] = train_ds

    # Validační data
    if val_path is not None:
        val_records = load_records(val_path)
        val_ds = Dataset.from_list(val_records, features=features)
        ds_dict["val"] = val_ds

    return DatasetDict(ds_dict)

# ---------- použití ----------

train_root = "/content/data/"

dataset = build_dataset(train_root)

print(dataset)
print(dataset["train"][0])

if "val" in dataset:
    print(dataset["val"][0])

DatasetDict({
    train: Dataset({
        features: ['file_name', 'ner_tags', 'bboxes', 'tokens', 'image'],
        num_rows: 39
    })
})
{'file_name': 'manually_labeled_2026-02-15_17-12-42.png', 'ner_tags': [0, 0, 0, 0, 29, 0, 0, 0, 25, 0, 0, 3, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 0, 19, 0, 21, 0, 23, 0, 0, 0, 27, 0, 0, 25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 29, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 29, 0], 'bboxes': [[81, 47, 273, 71], [285, 47, 443, 71], [81, 98, 228, 127], [243, 98, 368, 134], [382, 98, 499, 132], [514, 98, 580, 134], [593, 98, 796, 126], [810, 98, 999, 134], [1014, 98, 1222, 127], [80, 177, 183, 207], [553, 177, 595, 203], [925, 181, 102

#DATA VE FORMÁTU COCO

In [ ]:
import os
import json
from datasets import Dataset, DatasetDict, Image as HFImage
from collections import defaultdict

def load_coco_records(data_root_folder_path: str):
    metadata_path = os.path.join(data_root_folder_path, "metadata_coco.json")

    if not os.path.exists(metadata_path):
        return []

    with open(metadata_path, "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    # Vytvoříme mapování image_id -> list anotací
    ann_map = defaultdict(list)

    # Resetujeme ID anotací na malá, bezpečná čísla (0, 1, 2...)
    # Arrow nemá rád UUID ani stringy v číselných polích
    for idx, ann in enumerate(coco_data["annotations"]):
        image_id = ann["image_id"]

        ann_map[image_id].append({
            "id": idx,  # Globální index v rámci souboru (bezpečné pro Arrow)
            "image_id": image_id,
            "category_id": int(ann["category_id"]),
            "iscrowd": 0,
            "area": float(ann.get("area", 0.0)),
            "bbox": [float(x) for x in ann["bbox"]],
            "segmentation": ann.get("segmentation", [])
        })

    records = []
    for img in coco_data["images"]:
        img_id = img["id"]

        records.append({
            "image": os.path.join(data_root_folder_path, img["file_name"].split("/")[-1]),
            "image_id": int(img_id),
            "file_name": img["file_name"].split("/")[-1],
            "width": int(img.get("width", 0)),
            "height": int(img.get("height", 0)),
            "annotations": ann_map[img_id]
        })

    return records

def build_dataset(train_path: str, val_path: str | None = None):
    ds_dict = {}

    # Načtení trénovacích dat
    train_records = load_coco_records(train_path)
    # Tady se dřív vyvolal OverflowError
    train_ds = Dataset.from_list(train_records)
    train_ds = train_ds.cast_column("image", HFImage())
    ds_dict["train"] = train_ds

    if val_path and os.path.exists(val_path):
        val_records = load_coco_records(val_path)
        val_ds = Dataset.from_list(val_records)
        val_ds = val_ds.cast_column("image", HFImage())
        ds_dict["val"] = val_ds

    return DatasetDict(ds_dict)

dataset = build_dataset("/content/data/train", "/content/data/validation")

#NAHRÁNÍ DATASETU

In [ ]:
dataset.push_to_hub("TomasFAV/RealDocumentInvoiceCzechNameEntityRecognition")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  99%|#########8| 13.6MB / 13.8MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/TomasFAV/RealDocumentInvoiceCzechNameEntityRecognition/commit/26124554d116c4a32f58eef2d0a93ea5f33ccbe7', commit_message='Upload dataset', commit_description='', oid='26124554d116c4a32f58eef2d0a93ea5f33ccbe7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/TomasFAV/RealDocumentInvoiceCzechNameEntityRecognition', endpoint='https://huggingface.co', repo_type='dataset', repo_id='TomasFAV/RealDocumentInvoiceCzechNameEntityRecognition'), pr_revision=None, pr_num=None)